In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import pickle
import seaborn as sns
from itertools import product
from scipy.interpolate import interp1d
from pathlib import Path
from tqdm import tqdm
from joblib import Parallel, delayed
from sklearn.linear_model import LinearRegression, RANSACRegressor
from sklearn.metrics import mean_squared_error, r2_score

from resample_images import (
    make_spatial_grid,
    resample_spatial_grid,
    resample_temporal_grid,
    interpolate_stage_position,
    add_interpolated_positions,
    find_keyframes
)
from aruco import ArUcoBoard, detect_aruco, plot_aruco_detections
from model import (
    fit_affine_mapping,
    predict_physical_coords,
    visualize_calibration_results,
    ransac_filter_outliers,
    visualize_ransac_results,
    fit_affine_mapping_ransac,
)

## Load metadata

In [ ]:
aruco_board = ArUcoBoard(arena_dim_mm=(48, 72), scale_mm=0.3)

In [ ]:
def index_images_metadata(aruco_scan_dir):
    # Scan image metadata
    behavior_image_files = sorted(
        list((aruco_scan_dir / "behavior_images/").glob("*.csv"))
    )

    _individual_dfs = []
    for path in behavior_image_files:
        _df = pd.read_csv(path)
        assert len(_df) == 3
        _df.loc[:, "path"] = str(path).replace(".csv", ".jpg")
        _df.loc[:, "channel"] = [0, 1, 2]
        _individual_dfs.append(_df)
    return pd.concat(_individual_dfs, ignore_index=True)

In [ ]:
def load_keyframe_images(keyframe_metadata_df):
    keyframe_images = {}
    for path in keyframe_metadata_df["path"].unique():
        sel = keyframe_metadata_df[keyframe_metadata_df["path"] == path]
        image_all_channels = plt.imread(path)
        for _, row in sel.iterrows():
            channel = row["channel"]
            keyframe_images[row["frameId"]] = image_all_channels[:, :, channel]
    return keyframe_images


In [ ]:
def extract_coordinates_dataframe(
    keyframe_metadata_df, aruco_detection_results, aruco_board
):
    data = {
        "stage_x_mm": [],
        "stage_y_mm": [],
        "pixel_x_px": [],
        "pixel_y_px": [],
        "physical_x_mm": [],
        "physical_y_mm": [],
    }
    for _, row in keyframe_metadata_df.iterrows():
        frame_id = row["frameId"]
        stage_x_mm = row["stage_pos_x_mm"]
        stage_y_mm = row["stage_pos_y_mm"]

        detection_res = aruco_detection_results[frame_id]
        for i, aruco_id in enumerate(detection_res["ids"]):
            physical_corners_pos = aruco_board.grid_id_to_corner_xy_mm(aruco_id)
            for j in range(4):
                pixel_x, pixel_y = detection_res["corners"][i, j, :]
                physical_x, physical_y = physical_corners_pos[j, :]
                data["stage_x_mm"].append(stage_x_mm)
                data["stage_y_mm"].append(stage_y_mm)
                data["pixel_x_px"].append(pixel_x)
                data["pixel_y_px"].append(pixel_y)
                data["physical_x_mm"].append(physical_x)
                data["physical_y_mm"].append(physical_y)

    return pd.DataFrame(data)

In [ ]:
def run_calibration(base_dir, aruco_board, grid_stride=2):
    # Load stage position and image metadata
    images_metadata_df = index_images_metadata(base_dir)
    stage_position_log = pd.read_csv(base_dir / "stage_position/stage_position.csv")

    # Resample spatially and temporally; find keyframes
    keyframe_metadata_df = find_keyframes(
        stage_position_log, images_metadata_df, grid_stride
    )

    # Load keyframes and detect codes
    keyframe_images = load_keyframe_images(keyframe_metadata_df)
    aruco_detection_results = {}
    for _, row in tqdm(
        keyframe_metadata_df.iterrows(), total=len(keyframe_metadata_df)
    ):
        frame_id = row["frameId"]
        image = keyframe_images[frame_id]
        ids, corners = detect_aruco(image, horizontal_flip=True)
        aruco_detection_results[frame_id] = {"ids": ids, "corners": corners}

    # Extract coordinates dataframe
    coordinates_df = extract_coordinates_dataframe(
        keyframe_metadata_df, aruco_detection_results, aruco_board
    )

    # # Fit affine mapping
    # calibration_model = fit_affine_mapping(coordinates_df)
    ransac_result = ransac_filter_outliers(coordinates_df)
    models = {"x": ransac_result["models"]["x"], "y": ransac_result["models"]["y"]}

    return models, ransac_result, coordinates_df

In [ ]:
models_rbr, ransac_result_rbr, coordinates_df_rbr = run_calibration(
    Path.home() / "Spotlight/calibration/aruco_scan/row_by_row", aruco_board
)
models_cbc, ransac_result_cbc, coordinates_df_cbc = run_calibration(
    Path.home() / "Spotlight/calibration/aruco_scan/column_by_column", aruco_board
)

In [ ]:
visualize_ransac_results(coordinates_df_rbr, ransac_result_rbr)

In [ ]:
visualize_ransac_results(coordinates_df_cbc, ransac_result_cbc)

In [ ]:
y_model = ransac_result_rbr["models"]["y"]
x_model = ransac_result_cbc["models"]["x"]

affine_transform_mat = np.array(
    [
        [*x_model.coef_.squeeze(), x_model.intercept_],
        [*y_model.coef_.squeeze(), y_model.intercept_],
    ]
)

In [ ]:
affine_transform_mat

In [ ]:
def A_to_B(A):
    M = A[:, 2:4]  # row and column weights
    M_inv = np.linalg.inv(M)
    A_stage = A[:, 0:2]  # stage position
    A_bias = A[:, 4:5]  # bias
    B_stage = -M_inv @ A_stage
    B_physical = M_inv
    B_bias = -M_inv @ A_bias
    B = np.hstack([B_stage, B_physical, B_bias])
    return B

In [ ]:
affine_transform_mat_physical_to_pixel = A_to_B(affine_transform_mat)

In [ ]:
affine_transform_mat_physical_to_pixel

In [ ]:
affine_transform_mat_physical_to_pixel @ (
    np.array([105, 55, 5, 8, 1]).reshape(-1, 1)
)

In [ ]:
affine_transform_mat_physical_to_pixel.shape

In [ ]:
print("Calibration Metrics:")
print(f"R² for X: {calibration_model_column_by_column['metrics']['r2_x']:.4f}")
print(f"R² for Y: {calibration_model_column_by_column['metrics']['r2_y']:.4f}")
print(f"RMSE for X: {calibration_model_column_by_column['metrics']['rmse_x']:.4f} mm")
print(f"RMSE for Y: {calibration_model_column_by_column['metrics']['rmse_y']:.4f} mm")

visualize_calibration_results(
    coordinates_df_column_by_column, calibration_model_column_by_column, vmax=2
)

In [ ]:
def run_calibration_with_ransac(base_dir, aruco_board, grid_stride=2, verbose=True):
    # Load stage position and image metadata
    images_metadata_df = index_images_metadata(base_dir)
    stage_position_log = pd.read_csv(base_dir / "stage_position/stage_position.csv")

    # Resample spatially and temporally; find keyframes
    keyframe_metadata_df = find_keyframes(
        stage_position_log, images_metadata_df, grid_stride
    )

    # Load keyframes and detect codes
    keyframe_images = load_keyframe_images(keyframe_metadata_df)
    aruco_detection_results = {}
    for _, row in tqdm(
        keyframe_metadata_df.iterrows(), total=len(keyframe_metadata_df)
    ):
        frame_id = row["frameId"]
        image = keyframe_images[frame_id]
        ids, corners = detect_aruco(image, horizontal_flip=True)
        aruco_detection_results[frame_id] = {"ids": ids, "corners": corners}

    # Extract coordinates dataframe
    coordinates_df = extract_coordinates_dataframe(
        keyframe_metadata_df, aruco_detection_results, aruco_board
    )

    # Run RANSAC
    ransac_results = ransac_filter_outliers(
        coordinates_df,
        max_trials=1000,
        residual_threshold=1.0,  # 1mm threshold for outliers
    )
    if verbose:
        # Print basic metrics
        metrics = ransac_results["metrics"]
        print("\n--- RANSAC Filtering Results ---")
        print(f"Total points: {metrics['n_total']}")
        print(
            f"Inliers: {metrics['n_inliers']} ({100 - metrics['outlier_percent']:.1f}%)"
        )
        print(f"Outliers: {metrics['n_outliers']} ({metrics['outlier_percent']:.1f}%)")
        filtered_df = ransac_results["filtered_df"]

    return filtered_df, ransac_results

In [ ]:
calibration_model_with_ransac, coordinates_df_with_ransac, ransac_results = (
    run_calibration_with_ransac(aruco_scan_basedir, aruco_board)
)

In [ ]:
visualize_ransac_results(coordinates_df_row_by_row, ransac_results)

In [ ]:
print("Calibration Metrics:")
print(f"R² for X: {calibration_model_row_by_row['metrics']['r2_x']:.4f}")
print(f"R² for Y: {calibration_model_row_by_row['metrics']['r2_y']:.4f}")
print(f"RMSE for X: {calibration_model_row_by_row['metrics']['rmse_x']:.4f} mm")
print(f"RMSE for Y: {calibration_model_row_by_row['metrics']['rmse_y']:.4f} mm")

visualize_calibration_results(
    coordinates_df_with_ransac, calibration_model_with_ransac, vmax=2
)

In [ ]:
overall_model_y = 